# MGS-KAN при одинаковом бюджете параметров
Airfoil Self-Noise и Combined Cycle Power Plant, все исходные признаки. Сравниваем два скрытых ReLU-слоя MLP, четыре скрытых слоя Deep MLP, два блока ResNet и MGS-KAN с одиночными и парными гауссовыми ядрами. На каждой ступени (1 000, 4 000, 16 000) заранее зафиксирована одна конфигурация на семейство. Допуск 5% проверяется и относительно цели, и между крайними размерами в группе.

На фолде 0 из текущего эксперимента MLflow загружаются 36 компактных запусков предыдущей сетки. Если не хватает чекпоинтов, профиль укажет серию до начала обучения. Для трёх новых бюджетов запланированы 72 комбинации. Подходящие завершённые запуски также переиспользуются. Метрика — RMSE на валидации в лучшую эпоху. Три пары seed одинаковы для всех моделей. Тест остаётся за пределами этого эксперимента.

In [ ]:
PROJECT_DIR = None
DEVICE = "cuda:1"  # Выбери свободную GPU в Jupyter.
VERSION = "parameter-budget-v1"
FOLD = 0  # Для независимой проверки выбери другой доступный фолд.
DOWNLOAD = False  # Данные первого прогона уже должны быть на сервере.
GPU_BUDGET_MIB = 4096
CPU_THREADS = 1

In [ ]:
import sys
from pathlib import Path

if PROJECT_DIR is None:
    cwd = Path.cwd().resolve()
    checkout = next(
        (folder for folder in (cwd, *cwd.parents) if (folder / "src/experiments/runner.py").is_file()),
        None,
    )
else:
    checkout = Path(PROJECT_DIR).expanduser()
    if not checkout.is_absolute() or not (checkout / "src/experiments/runner.py").is_file():
        raise ValueError("Укажи в PROJECT_DIR абсолютный путь к корню mgs-kan на сервере.")
    checkout = checkout.resolve()

if checkout is not None and sys.path[0:1] != [str(checkout)]:
    sys.path.insert(0, str(checkout))

try:
    from src.paths import DATA_DIR, ENV_FILE, PROJECT_ROOT
except ModuleNotFoundError as error:
    raise ImportError("Укажи PROJECT_DIR и выполни ячейки заново.") from error

if checkout is not None and PROJECT_ROOT != checkout:
    raise RuntimeError("Ядро уже загрузило src из другой папки. Перезапусти ядро.")
if not (PROJECT_ROOT / "notebooks/baselines/run_series.ipynb").is_file():
    raise RuntimeError("Укажи PROJECT_DIR с рабочей копией mgs-kan и перезапусти ядро.")
print(f"Проект: {PROJECT_ROOT}\nНастройки: {ENV_FILE}\nДанные: {DATA_DIR}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

from src.config import MonitorSettings, ResourceSettings, TrackingSettings
from src.data import dataset_class
from src.experiments.search import ValidationGrid
from src.models import make_model, parameter_count

grid = ValidationGrid.preset("parameter_budget")
if FOLD != 0:
    # Компактные точки первого прогона относятся только к фолду 0.
    stages = tuple(s for s in grid.stages if not s.history_only)
    grid = ValidationGrid.model_validate({**grid.model_dump(), "stages": stages, "folds": (FOLD,)})
resources = ResourceSettings(_env_file=ENV_FILE, max_parallel_trials=1, cpu_threads=CPU_THREADS,
                             gpu_budget_mib=GPU_BUDGET_MIB)
tracking = TrackingSettings(_env_file=ENV_FILE)
monitoring = MonitorSettings(_env_file=ENV_FILE)
output = PROJECT_ROOT / "results" / "parameter-budget" / VERSION / f"fold-{FOLD}"
print(f"{grid.size} комбинаций · {DEVICE} · фолд {FOLD} · {len(grid.seeds)} пары seed")

In [ ]:
rows = []
for dataset, _, _, stage, settings in grid.plan():
    inputs = len(dataset_class(dataset).columns)
    options = settings.model_copy(update={"features": inputs, "init_seed": 0})
    for model in stage.models:
        count = parameter_count(make_model(model, options))
        rows.append(dict(dataset=dataset, stage=stage.name, model=model, expected_parameters=count,
                         target=stage.parameter_budget, size_group=str(stage.parameter_budget or stage.name)))
plan = pd.DataFrame(rows)
for (dataset, target), group in plan.dropna(subset=["target"]).groupby(["dataset", "target"]):
    counts = group.expected_parameters
    if (counts.sub(target).abs() > 0.05 * target).any() or counts.max() > 1.05 * counts.min():
        raise ValueError(f"{dataset}, бюджет {target:g}: допуск 5% превышен")
display(plan.pivot(index=["dataset", "size_group"], columns="model", values="expected_parameters")
        .astype("Int64"))

In [ ]:
runs = grid.run(version=f"{VERSION}-f{FOLD}", device=DEVICE, root=DATA_DIR, download=DOWNLOAD,
                tracking=tracking, monitoring=monitoring, resources=resources)
runs = runs.merge(plan, on=["dataset", "stage", "model"], validate="many_to_one")
if len(runs) != grid.size or not runs.parameters.eq(runs.expected_parameters).all():
    raise ValueError("Состав запусков или число обучаемых параметров расходится с планом")
output.mkdir(parents=True, exist_ok=True)
runs.to_csv(output / "runs.csv", index=False)
print(f"Готово {len(runs)}/{grid.size}. Из MLflow {runs.reused.sum()}. Результаты: {output}")

## Ошибка, размер и длительность
Слева — средняя валидационная RMSE и выборочное стандартное отклонение между тремя парами seed. Справа — та же ошибка в зависимости от среднего времени запуска по метрике MLflow `duration_seconds` (включает подготовку данных и трекинг). Время повторно использованных запусков берётся из исходного обучения.

Для графика времени берём только полные тройки seed с той же моделью GPU, одним одновременным обучением и тем же числом CPU-потоков. Исторические запуски с неизвестными условиями остаются в графике размера и выгрузке. Загрузка общей GPU другими пользователями может влиять на время.

In [ ]:
keys = ["dataset", "size_group", "model", "parameters"]
summary = runs.groupby(keys, as_index=False).agg(
    runs=("run_id", "size"), rmse=("val_rmse", "mean"), sd=("val_rmse", "std"), reused=("reused", "sum"),
)
gpu_name = torch.cuda.get_device_name(DEVICE)
eligible = runs.loc[
    runs.device_model.eq(gpu_name)
    & pd.to_numeric(runs.max_parallel_trials, errors="coerce").eq(1)
    & pd.to_numeric(runs.cpu_threads, errors="coerce").eq(CPU_THREADS)
]
timing = eligible.groupby(keys, as_index=False).agg(timed=("run_id", "size"), seconds=("seconds", "mean"))
summary = summary.merge(timing, on=keys, how="left", validate="one_to_one")
summary["timed"] = summary.timed.fillna(0).astype(int)
summary.loc[summary.timed != len(grid.seeds), "seconds"] = float("nan")
summary = summary.sort_values(["dataset", "parameters", "model"])
summary.to_csv(output / "summary.csv", index=False)
display(summary.round({"rmse": 4, "sd": 4, "seconds": 1}))
print(f"Сопоставимое время: {summary.seconds.notna().sum()}/{len(summary)} точек · {gpu_name}")

In [ ]:
fig, axes = plt.subplots(len(grid.datasets), 2, figsize=(12, 4.5 * len(grid.datasets)), squeeze=False)
units = {"airfoil": "дБ", "power_plant": "МВт"}
for (left, right), dataset in zip(axes, grid.datasets):
    part = summary.loc[summary.dataset == dataset]
    for model, points in part.groupby("model"):
        points = points.sort_values("parameters")
        artist = left.errorbar(points.parameters, points.rmse, yerr=points.sd, fmt="o-", capsize=3, label=model)
        timed = points.dropna(subset=["seconds"])
        if not timed.empty:
            right.errorbar(timed.seconds, timed.rmse, yerr=timed.sd, fmt="o", capsize=3,
                           color=artist.lines[0].get_color(), label=model)
    left.set_xscale("log")
    left.set_xlabel("Обучаемые параметры (лог. шкала)")
    right.set_xlabel("Среднее время одного запуска, с")
    for ax in (left, right):
        ax.set_title(dataset)
        ax.set_ylabel(f"RMSE валидации, {units[dataset]}")
        ax.grid(alpha=0.25)
        if ax.containers:
            ax.legend()
        else:
            ax.text(0.5, 0.5, "Нет троек с сопоставимым временем", transform=ax.transAxes, ha="center")
fig.tight_layout()
fig.savefig(output / "rmse-by-size-and-time.png", dpi=180, bbox_inches="tight")
plt.show()